In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver      
from selenium.webdriver.common.by import By
import time 
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

In [2]:
'''url = 'https://www.dges.gov.pt/guias/indcurso.asp'
browser = webdriver.Chrome(options=Options())  # Initialize a Chrome browser instance with options
browser.get(url)  # Open the URL in the browser
time.sleep(1)  '''

"url = 'https://www.dges.gov.pt/guias/indcurso.asp'\nbrowser = webdriver.Chrome(options=Options())  # Initialize a Chrome browser instance with options\nbrowser.get(url)  # Open the URL in the browser\ntime.sleep(1)  "

In [3]:
'''def scrape_current_letter(browser, course_institution_data):
    """Scrape all courses & institutions from the current letter page."""
    WebDriverWait(browser, 20).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))
    )

    soup = BeautifulSoup(browser.page_source, "html.parser")
    all_blocks = soup.find_all(["div"], class_=["box10", "lin-curso"])
    current_course = None

    for block in all_blocks:
        classes = block.get("class", [])

        # --- Course block ---
        if "box10" in classes:
            name_tag = block.find("div", class_="lin-area-c2")
            if name_tag:
                current_course = name_tag.text.strip()
                print(f"\n📘 Course: {current_course}")

        # --- Institution block ---
        elif "lin-curso" in classes and current_course:
            link_tag = block.find("a")
            if not link_tag:
                continue

            institution = link_tag.text.strip()
            href = link_tag.get("href")

            if any(kw in institution for kw in ["Universidade", "Instituto", "Escola", "Politécnico"]):
                print(f"🏫 Institution: {institution}")

                try:
                    # Click the institution
                    inst_element = WebDriverWait(browser, 10).until(
                        EC.element_to_be_clickable((By.XPATH, f"//a[@href='{href}']"))
                    )
                    browser.execute_script("arguments[0].scrollIntoView(true);", inst_element)
                    time.sleep(0.3)
                    inst_element.click()

                    # Wait for detail page
                    WebDriverWait(browser, 15).until(
                        EC.presence_of_all_elements_located((By.CLASS_NAME, "inside2"))
                    )
                    time.sleep(1)

                    # Parse the detail page
                    detail_soup = BeautifulSoup(browser.page_source, "html.parser")

                    # --- Extract Google Maps link ---
                    google_map = ""
                    inside_block = detail_soup.find("div", class_="inside2")
                    if inside_block:
                        map_link = inside_block.find("a", href=True, string=lambda t: t and "Mapa" in t)
                        if not map_link:
                            map_span = inside_block.find("span", class_="vislink", string=lambda t: "Mapa" in t)
                            if map_span and map_span.parent.name == "a":
                                map_link = map_span.parent
                        if map_link:
                            google_map = map_link["href"].strip()

                    print(f"🗺️ Google Maps link: {google_map if google_map else 'Not found'}")

                except Exception as e:
                    print(f"⚠️ Error scraping {institution}: {e}")
                    google_map = ""

                # Go back to the list page
                browser.back()
                time.sleep(1)
                WebDriverWait(browser, 20).until(
                    EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))
                )

                # Store
                course_institution_data.append((current_course, institution, href, google_map))

    return course_institution_data


# --- Step 1: Scrape letter A (already selected) ---
print("\n🔤 Scraping letter: A (default)")
course_institution_data = []
course_institution_data = scrape_current_letter(browser, course_institution_data)

# --- Step 2: Click and scrape all other letters ---
letters = browser.find_elements(By.CSS_SELECTOR, "div.noprint a")
letter_links = [(a.text.strip(), a.get_attribute("href")) for a in letters if a.text.strip()]

for letter, link in letter_links:
    print(f"\n🔤 Scraping letter: {letter}")
    browser.get(link)
    time.sleep(1.5)
    course_institution_data = scrape_current_letter(browser, course_institution_data)

# --- Save results ---
df = pd.DataFrame(course_institution_data, columns=["Course", "Institution", "Link", "GoogleMaps"])'''

'def scrape_current_letter(browser, course_institution_data):\n    """Scrape all courses & institutions from the current letter page."""\n    WebDriverWait(browser, 20).until(\n        EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))\n    )\n\n    soup = BeautifulSoup(browser.page_source, "html.parser")\n    all_blocks = soup.find_all(["div"], class_=["box10", "lin-curso"])\n    current_course = None\n\n    for block in all_blocks:\n        classes = block.get("class", [])\n\n        # --- Course block ---\n        if "box10" in classes:\n            name_tag = block.find("div", class_="lin-area-c2")\n            if name_tag:\n                current_course = name_tag.text.strip()\n                print(f"\n📘 Course: {current_course}")\n\n        # --- Institution block ---\n        elif "lin-curso" in classes and current_course:\n            link_tag = block.find("a")\n            if not link_tag:\n                continue\n\n            institution = link_tag.text.strip(

In [4]:
options = Options()
options.add_argument("--headless")
driver = webdriver.Chrome(options=options)

url = "https://www.mastersportal.com/search/master/portugal"
driver.get(url)

# keep scrolling until no new items load
last_count = 0

while True:
    # scroll to bottom
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)  # wait for new results to load
    
    titles = driver.find_elements(By.CSS_SELECTOR, "h2.StudyName")
    new_count = len(titles)
    
    if new_count == last_count:
        break   # no more new items
    last_count = new_count

# extract text
master_names = [t.text for t in titles]

driver.quit()

# save to pandas
df = pd.DataFrame(master_names, columns=["Master Name"])
print(df)


Empty DataFrame
Columns: [Master Name]
Index: []
